# Actividad 1 · Comparación entre grupos

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)

> Realizá solamente uno de los dos recorridos equivalentes disponibles: **SQL + Python** o **SQL + R**.

En la Semana 9 describiste variables por separado. Ahora vas a comparar dos grupos para responder una pregunta exploratoria: **¿los primeros bebés presentan diferencias en la duración del embarazo o en el peso al nacer respecto de los demás?**

El propósito no es demostrar una causa, sino aprender a reconocer diferencias y valorar su importancia en contexto.

## Preparación automática

Ejecutá la celda siguiente sin modificarla. Preparará el dataset, las variables necesarias y la conexión con DuckDB. Esta preparación técnica no forma parte de los contenidos evaluados.

In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AllenDowney/ThinkStats/v3/data/2002FemPreg.dat.gz"
DATA_PATH = Path("2002FemPreg.dat.gz")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

colspecs = [
    (0, 12), (12, 14), (56, 58), (58, 60), (274, 276),
    (276, 277), (277, 279), (283, 287), (422, 440),
]
names = [
    "caseid", "pregordr", "birthwgt_lb", "birthwgt_oz", "prglngth",
    "outcome", "birthord", "agepreg", "finalwgt",
]

embarazos = pd.read_fwf(
    DATA_PATH, names=names, colspecs=colspecs, compression="gzip"
)
embarazos["birthwgt_lb"] = embarazos["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)
embarazos["birthwgt_oz"] = embarazos["birthwgt_oz"].replace([97, 98, 99], np.nan)
embarazos["totalwgt_lb"] = embarazos["birthwgt_lb"] + embarazos["birthwgt_oz"] / 16
embarazos["totalwgt_kg"] = (embarazos["totalwgt_lb"] * 0.45359237).round(3)
embarazos["agepreg"] = embarazos["agepreg"] / 100
embarazos = embarazos[[
    "caseid", "pregordr", "prglngth", "outcome", "birthord",
    "birthwgt_lb", "birthwgt_oz", "totalwgt_lb", "totalwgt_kg",
    "agepreg", "finalwgt",
]]

con = duckdb.connect(":memory:")
con.register("embarazos_df", embarazos)
con.execute("CREATE OR REPLACE TABLE embarazos AS SELECT * FROM embarazos_df")
print(f"Entorno preparado: {len(embarazos):,} registros disponibles en la tabla 'embarazos'.")

## Cantidad de casos en cada grupo

Trabajaremos únicamente con nacimientos vivos (`outcome = 1`) y diferenciaremos:

- **Primeros bebés:** `birthord = 1`.
- **Otros bebés:** `birthord > 1`.

La primera consulta permite conocer el tamaño de cada grupo.

In [ ]:
con.sql("""SELECT
    CASE WHEN birthord = 1 THEN 'Primeros' ELSE 'Otros' END AS grupo,
    COUNT(*) AS cantidad
FROM embarazos
WHERE outcome = 1
GROUP BY grupo
ORDER BY grupo""")

## Comparación de la duración del embarazo

La media resume el centro de cada grupo y el desvío estándar ayuda a interpretar cuánto varían los casos dentro de cada uno.

In [ ]:
duracion_resumen = con.sql("""SELECT
    CASE WHEN birthord = 1 THEN 'Primeros' ELSE 'Otros' END AS grupo,
    COUNT(*) AS cantidad,
    ROUND(AVG(prglngth), 2) AS promedio_semanas,
    ROUND(STDDEV(prglngth), 2) AS desvio_semanas
FROM embarazos
WHERE outcome = 1
GROUP BY grupo
ORDER BY grupo""").df()
duracion_resumen

Una diferencia entre promedios debe leerse junto con la variabilidad de los grupos. Para observar el resultado completo, representaremos las distribuciones.

In [ ]:
duracion_distribucion = con.sql("""SELECT
    prglngth AS semanas,
    CASE WHEN birthord = 1 THEN 'Primeros' ELSE 'Otros' END AS grupo,
    COUNT(*) AS cantidad
FROM embarazos
WHERE outcome = 1 AND prglngth BETWEEN 27 AND 46
GROUP BY semanas, grupo
ORDER BY semanas, grupo""").df()
duracion_distribucion.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
for grupo, color in [("Primeros", "#1565c0"), ("Otros", "#ff9800")]:
    datos_grupo = duracion_distribucion[duracion_distribucion["grupo"] == grupo]
    ax.plot(
        datos_grupo["semanas"], datos_grupo["cantidad"],
        marker="o", linewidth=2, label=grupo, color=color
    )

ax.set_xlabel("Duración del embarazo (semanas)")
ax.set_ylabel("Cantidad")
ax.set_title("Duración del embarazo por grupo")
ax.legend()
plt.show()

## Comparación del peso al nacer

Usaremos `totalwgt_kg` para expresar el peso en kilogramos y mantener la misma unidad empleada durante la Semana 9.

In [ ]:
con.sql("""SELECT
    CASE WHEN birthord = 1 THEN 'Primeros' ELSE 'Otros' END AS grupo,
    COUNT(totalwgt_kg) AS cantidad,
    ROUND(AVG(totalwgt_kg), 2) AS promedio_kg,
    ROUND(STDDEV(totalwgt_kg), 2) AS desvio_kg
FROM embarazos
WHERE outcome = 1
GROUP BY grupo
ORDER BY grupo""")

## Ejercicio

Compará la **edad de la madre** (`agepreg`) entre primeros bebés y otros. Escribí una consulta SQL que muestre la cantidad de casos, el promedio y el desvío estándar de la edad para cada grupo. Después redactá dos o tres oraciones que respondan:

1. ¿Qué diferencia observás entre los promedios?
2. ¿La diferencia parece grande en relación con la variabilidad de los grupos?
3. ¿Qué no podrías concluir todavía?

In [ ]:
# Escribí acá la consulta SQL del ejercicio
con.sql("""

""")

**Interpretación:** escribí acá tu respuesta.

## Cierre

Comparar grupos permite reconocer diferencias, pero una diferencia observada no demuestra por sí sola una relación causal ni garantiza relevancia práctica. En la siguiente actividad vas a profundizar esta distinción.

**Bibliografía:** Downey, A. B. (2025). *Think Stats* (3.ª ed.), capítulo 1.

In [ ]:
con.close()